# Auditoria BCBI × ODBC × SAGI

Compara os KPIs exibidos no **BCBI** (dashboard da empresa) com os dados brutos do **ODBC** (`base.csv`)  
e das **movimentações operacionais do SAGI** (`SYG_MOV COMPRAS.csv` / `SYG_MOV VENDAS.csv`).

**Divisão:** Seletiva (`1.2.x` despesa / `2.2.x` receita)  
**Período:** detectado automaticamente — mês corrente

---

## Como usar

1. Abra o BCBI, filtre por **Seletiva** e pelo **mês corrente**
2. Preencha o dicionário `bcbi` na **Célula 2** com os valores que você vê no dashboard
3. Execute todas as células em sequência (`Run All`)
4. Veja o **veredicto colorido** e o **detalhamento das divergências** nas últimas células

> **Tolerância padrão:** `R$ 1.000,00` — altere `TOLERANCIA` na Célula 1 se necessário.

In [ ]:
# ─── Imports e configuração ────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
from datetime import date
from IPython.display import display, HTML

# Caminhos: detecta automaticamente se o notebook roda da raiz ou de 04-Notebooks/
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, '02-Referencias')):
    WORKSPACE = _cwd                                    # CWD = raiz do workspace
elif os.path.isdir(os.path.join(_cwd, '..', '02-Referencias')):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, '..'))  # CWD = 04-Notebooks/
else:
    raise FileNotFoundError(f'Pasta 02-Referencias nao encontrada a partir de: {_cwd}')

REFS         = os.path.join(WORKSPACE, '02-Referencias')
PATH_ODBC    = os.path.join(REFS, 'base.csv')
PATH_COMPRAS = os.path.join(REFS, 'SYG_MOV COMPRAS.csv')
PATH_VENDAS  = os.path.join(REFS, 'SYG_MOV VENDAS.csv')

# Período: mês corrente
hoje      = date.today()
MES       = f'{hoje.year}-{hoje.month:02d}'   # ex: '2026-03'
MES_LABEL = hoje.strftime('%m/%Y')            # ex: '03/2026'

# Tolerância de diferença aceitável (R$)
TOLERANCIA = 1_000.00

print(f'Período de análise : {MES_LABEL}  ({MES}-xx)')
print(f'Tolerância         : R$ {TOLERANCIA:,.2f}')
print(f'ODBC               : {PATH_ODBC}')
print(f'MOV COMPRAS        : {PATH_COMPRAS}')
print(f'MOV VENDAS         : {PATH_VENDAS}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#   PREENCHA AQUI OS VALORES DO BCBI
#   (Dashboard → Seletiva → mês corrente)
#
#   Linha 1 do dashboard:  RECEITAS TOTAIS | COMPRAS | CUSTOS TOTAIS | ESTOQUE | RESULTADO
#   Linha 2 do dashboard:  VENDAS DE SUCATA (R$ e KG) | COMPRAS DE SUCATA (R$ e KG)
# ══════════════════════════════════════════════════════════════════════════════

bcbi = {
    # ── Linha 1 ───────────────────────────────────────────────────────────────
    'receitas_totais_rs':    9_362_592.47,   # card RECEITAS TOTAIS
    'compras_financeiro_rs': 4_551_273.45,   # card COMPRAS (V. Bruto)
    'custos_totais_rs':      2_461_460.78,   # card CUSTOS TOTAIS
    'estoque_rs':            1_240_667.10,   # card ESTOQUE  ← calculado pelo BCBI, sem fonte ODBC direta
    'resultado_rs':          1_109_191.15,   # card RESULTADO

    # ── Linha 2 ───────────────────────────────────────────────────────────────
    'vendas_sucata_rs':      8_276_156.35,   # card VENDAS DE SUCATA  R$
    'vendas_sucata_kg':      6_448_453,      # card VENDAS DE SUCATA  KG
    'compras_sucata_rs':     4_454_514.05,   # card COMPRAS DE SUCATA R$
    'compras_sucata_kg':     5_593_948,      # card COMPRAS DE SUCATA KG
}

print(f"BCBI preenchido em {hoje.strftime('%d/%m/%Y')} — {len(bcbi)} métricas")

In [ ]:
# ─── Carga e filtragem dos dados ────────────────────────────────────────────

def _to_float(series):
    """Converte coluna numérica com vírgula decimal (padrão BR) para float."""
    return pd.to_numeric(
        series.astype(str).str.strip().str.replace(',', '.', regex=False),
        errors='coerce'
    )

# ── ODBC (base.csv) ──────────────────────────────────────────────────────────
print('Carregando base.csv ...')
df = pd.read_csv(PATH_ODBC, sep=';', encoding='latin-1', dtype=str)
df = df.dropna(subset=['codcen'])                            # remove linha vazia final
df['valor_bruto'] = _to_float(df['valor_bruto'])
df['valor_plano'] = _to_float(df['valor_plano'])

# Filtro de mês pelo vencimento (ite_pagrec_vencimento = 'YYYY-MM-DD')
df_mes  = df[df['ite_pagrec_vencimento'].str.startswith(MES, na=False)].copy()

# Filtro Seletiva: CC 1.2.x (despesa) e 2.2.x (receita)
df_desp = df_mes[df_mes['codcen'].str.startswith('1.2', na=False)].copy()
df_rec  = df_mes[df_mes['codcen'].str.startswith('2.2', na=False)].copy()
print(f'  Registros Seletiva Despesa : {len(df_desp):,}')
print(f'  Registros Seletiva Receita : {len(df_rec):,}')

# ── SAGI MOV COMPRAS ─────────────────────────────────────────────────────────
print('\nCarregando SYG_MOV COMPRAS.csv ...')
df_compras = pd.read_csv(PATH_COMPRAS, sep=';', encoding='latin-1', dtype=str)
df_compras['valor_total']   = _to_float(df_compras['valor_total'])
df_compras['n_qtde_fat_kg'] = _to_float(df_compras['n_qtde_fat_kg'])
df_compras = df_compras[df_compras['data'].str.startswith(MES, na=False)].copy()
df_compras = df_compras[df_compras['c_tipo_movimento'] == 'ENTRADA'].copy()
# Apenas filiais Seletiva (G3S Dourados, Londrina, Maringá, Prudente, CG, Cidade Alta, Assis)
df_compras = df_compras[df_compras['c_filial_carregamento'].str.startswith('G3S', na=False)].copy()
print(f'  Registros Compras Seletiva : {len(df_compras):,}')

# ── SAGI MOV VENDAS ──────────────────────────────────────────────────────────
print('\nCarregando SYG_MOV VENDAS.csv ...')
df_vendas = pd.read_csv(PATH_VENDAS, sep=';', encoding='latin-1', dtype=str)
df_vendas['valor_total']   = _to_float(df_vendas['valor_total'])
df_vendas['n_qtde_fat_kg'] = _to_float(df_vendas['n_qtde_fat_kg'])
df_vendas = df_vendas[df_vendas['data'].str.startswith(MES, na=False)].copy()
# SAIDA + DEV-SAIDA (devoluções têm valores negativos — entram na soma líquida)
df_vendas = df_vendas[df_vendas['c_tipo_movimento'].isin(['SAIDA', 'DEV-SAIDA'])].copy()
df_vendas = df_vendas[df_vendas['c_filial_carregamento'].str.startswith('G3S', na=False)].copy()
print(f'  Registros Vendas Seletiva  : {len(df_vendas):,}')

print('\nDados carregados com sucesso.')

In [ ]:
# ─── Métricas ODBC (base.csv) ───────────────────────────────────────────────

# Receitas Totais: todos os CCs 2.2.x, qualquer conta
odbc_receitas_totais  = df_rec['valor_bruto'].sum()

# Vendas de Sucata: CC 2.2.x, conta 4.1.1
odbc_vendas_sucata_rs = df_rec.loc[df_rec['codcdc'] == '4.1.1', 'valor_bruto'].sum()

# Compras Financeiro: CC 1.2.x, conta 6.1.1 (COMPRAS DE SUCATAS)
odbc_compras_fin_rs   = df_desp.loc[df_desp['codcdc'] == '6.1.1', 'valor_bruto'].sum()

# Custos Totais: CC 1.2.x, todas as contas EXCETO 6.1.1
# Inclui: frete (6.6.x), operacional (7.1.x), pessoal (7.3.x),
#         tributário (7.4.x), administrativo (7.5.x), pátio (7.6.x), reformas (7.7.x)
odbc_custos_totais    = df_desp.loc[~df_desp['codcdc'].isin(['6.1.1']), 'valor_bruto'].sum()

# Resultado estimado (sem ajuste de estoque — o BCBI aplica estoque que não está no ODBC)
odbc_resultado_semest = odbc_receitas_totais - odbc_compras_fin_rs - odbc_custos_totais

print(f'Métricas ODBC — Seletiva — {MES_LABEL}')
print(f'  Receitas Totais            : R$ {odbc_receitas_totais:>15,.2f}')
print(f'  Vendas Sucata (4.1.1)      : R$ {odbc_vendas_sucata_rs:>15,.2f}')
print(f'  Compras Financeiro (6.1.1) : R$ {odbc_compras_fin_rs:>15,.2f}')
print(f'  Custos Totais (excl. 6.1.1): R$ {odbc_custos_totais:>15,.2f}')
print(f'  Resultado (sem estoque)    : R$ {odbc_resultado_semest:>15,.2f}')

In [ ]:
# ─── Métricas SAGI (movimentações operacionais) ──────────────────────────────

sagi_compras_rs = df_compras['valor_total'].sum()
sagi_compras_kg = df_compras['n_qtde_fat_kg'].sum()
sagi_vendas_rs  = df_vendas['valor_total'].sum()
sagi_vendas_kg  = df_vendas['n_qtde_fat_kg'].sum()

# R$/kg médio para referência
cmp_rkg = sagi_compras_rs / sagi_compras_kg if sagi_compras_kg else 0
vnd_rkg = sagi_vendas_rs  / sagi_vendas_kg  if sagi_vendas_kg  else 0

print(f'Métricas SAGI — Seletiva — {MES_LABEL}')
print(f'  Compras Sucata (R$) : R$ {sagi_compras_rs:>15,.2f}   ({cmp_rkg:.2f} R$/kg)')
print(f'  Compras Sucata (KG) :    {sagi_compras_kg:>15,.0f} kg')
print(f'  Vendas  Sucata (R$) : R$ {sagi_vendas_rs:>15,.2f}   ({vnd_rkg:.2f} R$/kg)')
print(f'  Vendas  Sucata (KG) :    {sagi_vendas_kg:>15,.0f} kg')

In [ ]:
# ─── Tabela comparativa e veredicto ─────────────────────────────────────────

_NA = float('nan')

def _fmtv(v, kg=False):
    """Formata valor para exibição (R$ ou KG)."""
    if pd.isna(v):
        return '—'
    return f'{v:,.0f} kg' if kg else f'R$ {v:,.2f}'

def _fmtd(v):
    """Formata diferença com sinal."""
    if pd.isna(v):
        return '—'
    return f'{v:+,.2f}'

def _fmtp(diff, base):
    """Calcula e formata percentual de diferença."""
    if pd.isna(diff) or pd.isna(base) or base == 0:
        return '—'
    return f'{diff / base * 100:+.2f}%'

def _cor(v):
    """CSS de cor para célula de diferença — verde/amarelo/vermelho."""
    if pd.isna(v):
        return ''
    if abs(v) <= TOLERANCIA:
        return 'background-color:#c6efce;color:#276221'
    if abs(v) <= 10_000:
        return 'background-color:#ffeb9c;color:#9c5700'
    return 'background-color:#ffc7ce;color:#9c0006'

# ── Definição das métricas a comparar ────────────────────────────────────────
#  (label, eh_kg, val_bcbi, val_odbc, val_sagi)
linhas = [
    ('Receitas Totais (R$)',    False, bcbi['receitas_totais_rs'],    odbc_receitas_totais,  _NA            ),
    ('Vendas Sucata (R$)',      False, bcbi['vendas_sucata_rs'],      odbc_vendas_sucata_rs, sagi_vendas_rs ),
    ('Vendas Sucata (KG)',      True,  bcbi['vendas_sucata_kg'],      _NA,                   sagi_vendas_kg ),
    ('Compras Financeiro (R$)', False, bcbi['compras_financeiro_rs'], odbc_compras_fin_rs,   sagi_compras_rs),
    ('Compras Sucata (R$)',     False, bcbi['compras_sucata_rs'],     _NA,                   sagi_compras_rs),
    ('Compras Sucata (KG)',     True,  bcbi['compras_sucata_kg'],     _NA,                   sagi_compras_kg),
    ('Custos Totais (R$)',      False, bcbi['custos_totais_rs'],      odbc_custos_totais,    _NA            ),
    ('Estoque (R$)',            False, bcbi['estoque_rs'],            _NA,                   _NA            ),
    ('Resultado (R$)',          False, bcbi['resultado_rs'],          odbc_resultado_semest, _NA            ),
]

rows = []
for label, kg, b, o, s in linhas:
    d_bo = b - o if not (pd.isna(b) or pd.isna(o)) else _NA
    d_bs = b - s if not (pd.isna(b) or pd.isna(s)) else _NA
    rows.append({
        'Métrica':          label,
        'BCBI':             _fmtv(b, kg),
        'ODBC (base.csv)':  _fmtv(o, kg),
        'SAGI (MOV)':       _fmtv(s, kg),
        'Δ BCBI-ODBC':      d_bo,
        '% ODBC':           _fmtp(d_bo, o),
        'Δ BCBI-SAGI':      d_bs,
        '% SAGI':           _fmtp(d_bs, s),
    })

cmp = pd.DataFrame(rows)

# ── Estilo da tabela ─────────────────────────────────────────────────────────
th_style = 'background-color:#1f3864;color:white;font-weight:bold;padding:6px 10px'
td_style = 'padding:5px 10px;border:1px solid #ddd'
tr_even  = 'background-color:#f5f7fa'

styled = (
    cmp.style
       .format({'Δ BCBI-ODBC': _fmtd, 'Δ BCBI-SAGI': _fmtd}, na_rep='—')
       .set_table_styles([
           {'selector': 'th', 'props': th_style},
           {'selector': 'td', 'props': td_style},
           {'selector': 'tr:nth-child(even)', 'props': tr_even},
       ])
)

# Compatível com pandas 1.x (applymap) e 2.x (map)
try:
    styled = styled.map(_cor, subset=['Δ BCBI-ODBC', 'Δ BCBI-SAGI'])
except AttributeError:
    styled = styled.applymap(_cor, subset=['Δ BCBI-ODBC', 'Δ BCBI-SAGI'])

display(styled)

# ── Veredicto ────────────────────────────────────────────────────────────────
div_bo = cmp[cmp['Δ BCBI-ODBC'].notna() & (cmp['Δ BCBI-ODBC'].abs() > TOLERANCIA)]
div_bs = cmp[cmp['Δ BCBI-SAGI'].notna() & (cmp['Δ BCBI-SAGI'].abs() > TOLERANCIA)]
total_div = len(div_bo) + len(div_bs)
now_str   = hoje.strftime('%d/%m/%Y')

if total_div == 0:
    ok_style = ('background:#c6efce;color:#276221;padding:14px;border-radius:8px;'
                'font-size:1.1em;font-weight:bold;margin-top:10px')
    display(HTML(
        f'<div style="{ok_style}">'
        f'✅  AUDITORIA OK — Todos os valores batem '
        f'(tolerância R$ {TOLERANCIA:,.0f})  |  {now_str}'
        f'</div>'
    ))
else:
    err_style = ('background:#ffc7ce;color:#9c0006;padding:14px;border-radius:8px;'
                 'font-size:1.1em;font-weight:bold;margin-top:10px')
    items_html = ''
    for _, row in div_bo.iterrows():
        items_html += f'<li>BCBI vs ODBC — {row["Métrica"]}: Δ = {_fmtd(row["Δ BCBI-ODBC"])}</li>'
    for _, row in div_bs.iterrows():
        items_html += f'<li>BCBI vs SAGI — {row["Métrica"]}: Δ = {_fmtd(row["Δ BCBI-SAGI"])}</li>'
    display(HTML(
        f'<div style="{err_style}">'
        f'⚠️  DIVERGÊNCIAS ENCONTRADAS: {total_div} métrica(s)  |  {now_str}'
        f'<ul style="font-weight:normal;margin-top:8px">{items_html}</ul>'
        f'</div>'
    ))

In [ ]:
# ─── Detalhamento das divergências ──────────────────────────────────────────
#
# Para cada métrica com diferença acima da tolerância, exibe:
#   • Total por filial  →  ajuda a identificar qual filial está causando a divergência
#   • Top-10 maiores lançamentos  →  candidatos óbvios de reclassificação no SAGI

def _detalha_odbc(label, subset_df, prefixo=''):
    """Exibe resumo por filial e top-10 de um subconjunto do ODBC."""
    h4_style = 'color:#1f3864;margin-top:18px;margin-bottom:4px'
    display(HTML(f'<h4 style="{h4_style}">{prefixo}{label}</h4>'))

    por_filial = (
        subset_df.groupby('filial')['valor_bruto']
                 .sum().sort_values(ascending=False)
                 .reset_index()
    )
    por_filial.columns = ['Filial', 'Total R$']
    por_filial['Total R$'] = por_filial['Total R$'].map('R$ {:,.2f}'.format)
    display(por_filial.reset_index(drop=True))

    top10 = (
        subset_df.nlargest(10, 'valor_bruto')
                 [['ite_pagrec_vencimento', 'filial', 'codcdc', 'descdc', 'nome', 'valor_bruto']]
                 .copy()
    )
    top10.columns = ['Vencimento', 'Filial', 'Conta', 'Descrição', 'Favorecido', 'Valor R$']
    top10['Valor R$'] = top10['Valor R$'].map('R$ {:,.2f}'.format)
    display(top10.reset_index(drop=True))


def _detalha_sagi(label, df_mov, col_rs='valor_total', col_kg='n_qtde_fat_kg'):
    """Exibe resumo por filial de uma movimentação SAGI."""
    h4_style = 'color:#1f3864;margin-top:18px;margin-bottom:4px'
    display(HTML(f'<h4 style="{h4_style}">📦 {label} — por filial (SAGI)</h4>'))
    agg = (
        df_mov.groupby('c_filial_carregamento')
              .agg(rs=(col_rs, 'sum'), kg=(col_kg, 'sum'))
              .sort_values('rs', ascending=False)
              .reset_index()
    )
    agg.columns = ['Filial', 'Total R$', 'Total KG']
    agg['Total R$'] = agg['Total R$'].map('R$ {:,.2f}'.format)
    agg['Total KG'] = agg['Total KG'].map('{:,.0f} kg'.format)
    display(agg.reset_index(drop=True))


# ── Mapa de filtros ODBC para cada métrica ───────────────────────────────────
# (None no lugar de df = usa o dataframe completo sem filtro adicional)
_filtros_desp = {
    'Compras Financeiro (R$)': df_desp['codcdc'] == '6.1.1',
    'Custos Totais (R$)':      ~df_desp['codcdc'].isin(['6.1.1']),
}
_filtros_rec = {
    'Receitas Totais (R$)': df_rec['codcdc'].notna(),          # todas as contas
    'Vendas Sucata (R$)':   df_rec['codcdc'] == '4.1.1',
}

# ── Execução condicional ──────────────────────────────────────────────────────
if total_div == 0:
    print('Nenhuma divergência a detalhar — auditoria OK.')
else:
    sec_style = 'color:#9c0006;border-bottom:2px solid #9c0006;padding-bottom:4px'

    if len(div_bo) > 0:
        display(HTML(f'<h3 style="{sec_style}">🔍 Detalhamento BCBI vs ODBC</h3>'))
        for metrica in div_bo['Métrica']:
            if metrica in _filtros_desp:
                _detalha_odbc(metrica, df_desp[_filtros_desp[metrica]], '💰 ')
            elif metrica in _filtros_rec:
                _detalha_odbc(metrica, df_rec[_filtros_rec[metrica]], '📈 ')
            elif metrica == 'Resultado (R$)':
                display(HTML('<p><i>O Resultado é derivado — verifique individualmente '
                             'Receitas, Compras e Custos.</i></p>'))

    if len(div_bs) > 0:
        display(HTML(f'<h3 style="{sec_style}">🔍 Detalhamento BCBI vs SAGI</h3>'))
        for metrica in div_bs['Métrica']:
            if 'Venda' in metrica or 'venda' in metrica:
                _detalha_sagi(metrica, df_vendas)
            elif 'Compra' in metrica or 'compra' in metrica:
                _detalha_sagi(metrica, df_compras)